# Module 3 — Agent Trajectory Evaluation

Module 2's tool-use and task-completion metrics all score a **finished** interaction: given the tools that were called and the final answer, was it correct? This module asks a different question: given a **multi-step run**, was the *path* the agent took any good — not just where it ended up? An agent can reach the right final answer while taking twice as many steps as necessary, retrying a failed call blindly instead of correcting course, or repeating an identical tool call for no reason. None of Module 2's metrics see any of that, because they only look at the final `tools_called` list and `actual_output`, not the shape of the trace that produced them.

_Source: adapted from `04_Agent_RAG_Eval/rag_agent_eval_langgraph_new.ipynb` Parts 3–4 (mocked) and `rag_agent_eval_langgraph_openai.ipynb` Part 3 (real tool-calling agent)._

**Structure of this notebook:**
- **Part A** — a simple mocked agent loop, introducing step-wise trajectory scoring on a 2-tool task.
- **Part B** — the flagship content: a realistic 3-tool customer-support scenario, deliberately built to exercise **9 separate trace-level metrics** in one run.
- **Part C** — the same idea, with a real OpenAI tool-calling agent instead of a scripted mock planner.

## Part A — Mocked: a simple agent trajectory loop

**Approach:** run an agent tool-calling loop (`plan_and_act`, looping via a conditional edge) that builds a **trace**, then hand the trace to a separate `evaluate` node. Keeping the agent loop and the eval logic as separate nodes mirrors real practice — the agent doesn't grade itself; a separate harness does, against a **gold trajectory** defined ahead of time.

In [ ]:
# ============ IMPORTS ============
from typing import TypedDict, List, Dict, Optional
from langgraph.graph import StateGraph, END

print("Imports OK")

In [ ]:
# ============ STATE SCHEMA ============
class AgentEvalState(TypedDict):
    query: str
    gold_tool_sequence: List[str]     # ground truth trajectory, defined by the eval harness
    trace: List[Dict]                  # [{"tool", "args", "result", "used"}]
    current_step: int
    final_answer: Optional[str]
    task_success: Optional[bool]
    tool_selection_correct: List[bool]
    unnecessary_calls: List[str]

In [ ]:
# ============ MOCK TOOLS ============
def tool_search_docs(args):
    return f"docs about {args.get('topic', '?')}"

def tool_calculator(args):
    return str(eval(args.get("expr", "0")))  # toy only -- never eval() untrusted input in real code

def tool_send_email(args):
    return f"email sent to {args.get('to', '?')}"

TOOLS = {
    "search_docs": tool_search_docs,
    "calculator": tool_calculator,
    "send_email": tool_send_email,
}
print("Tools registered:", list(TOOLS.keys()))

In [ ]:
# ============ MOCK PLANNER ============
# In production this is an LLM call deciding the next action given state + tool results.
# This plan deliberately repeats a call, to demonstrate "unnecessary tool call" detection.
def mock_plan_next_step(state: AgentEvalState) -> Optional[Dict]:
    plan = [
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},  # redundant repeat
        {"tool": "calculator", "args": {"expr": "2**10"}},
    ]
    step = state["current_step"]
    return plan[step] if step < len(plan) else None

In [ ]:
# ============ GRAPH NODES ============
def plan_and_act_node(state: AgentEvalState) -> AgentEvalState:
    next_call = mock_plan_next_step(state)
    if next_call is None:
        state["final_answer"] = "Task complete."
        return state

    tool_name, args = next_call["tool"], next_call["args"]
    result = TOOLS[tool_name](args)

    # A call is "unnecessary" if the exact same (tool, args) already happened --
    # the result would be identical, so it added no new information.
    is_duplicate = any(t["tool"] == tool_name and t["args"] == args for t in state["trace"])

    state["trace"].append({
        "tool": tool_name, "args": args, "result": result, "used": not is_duplicate,
    })
    state["current_step"] += 1
    return state


def should_continue(state: AgentEvalState) -> str:
    # This is the loop: keep acting until the planner signals completion.
    return "plan_and_act" if state.get("final_answer") is None else "evaluate"


def evaluate_node(state: AgentEvalState) -> AgentEvalState:
    actual_tools = [t["tool"] for t in state["trace"]]
    gold_tools = state["gold_tool_sequence"]

    # Step-wise trajectory match against the gold sequence -- this is how you
    # evaluate MULTI-STEP tasks: score each step, not just the final answer.
    correctness = [
        (actual_tools[i] if i < len(actual_tools) else None) == gold_tools[i]
        for i in range(len(gold_tools))
    ]
    state["tool_selection_correct"] = correctness

    # Flags raised during the trace itself -- redundant calls with no new info.
    state["unnecessary_calls"] = [t["tool"] for t in state["trace"] if not t["used"]]

    # Task completion = reached a final answer AND every gold step was matched.
    # (In a real harness you'd also verify END STATE, e.g. did the email actually send.)
    state["task_success"] = (
        state["final_answer"] is not None and sum(correctness) == len(gold_tools)
    )
    return state

print("Nodes defined")

In [ ]:
# ============ BUILD & RUN THE GRAPH ============
builder = StateGraph(AgentEvalState)
builder.add_node("plan_and_act", plan_and_act_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("plan_and_act")
builder.add_conditional_edges(
    "plan_and_act",
    should_continue,
    {"plan_and_act": "plan_and_act", "evaluate": "evaluate"},
)
builder.add_edge("evaluate", END)

agent_eval_graph = builder.compile()

initial_state: AgentEvalState = {
    "query": "Look up BST delete complexity and compute 2^10",
    "gold_tool_sequence": ["search_docs", "calculator"],   # the IDEAL 2-step trajectory
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "task_success": None,
    "tool_selection_correct": [],
    "unnecessary_calls": [],
}

result = agent_eval_graph.invoke(initial_state)

print("=== Trace ===")
for i, step in enumerate(result["trace"]):
    print(f"  Step {i}: {step['tool']}({step['args']}) -> {step['result']!r}  | used={step['used']}")

print()
print("=== Eval ===")
print("Tool selection correctness vs gold:", result["tool_selection_correct"])
print("Unnecessary calls detected:        ", result["unnecessary_calls"])
print("Task success:                      ", result["task_success"])
print("Final answer:                      ", result["final_answer"])

**Reading the output:**
- **Tool selection accuracy** — step 0 matches gold (`search_docs`), step 1 doesn't (gold expected `calculator`, agent repeated `search_docs`) → `[True, False]`. This is exactly the "right tool, wrong step" signal a single-outcome metric like Module 2's `ToolCorrectnessMetric` can't give you — it only compares the *set* of tools called, not the sequence.
- **Unnecessary tool calls** — the duplicate `search_docs` call is flagged because its `(tool, args)` pair already appeared in the trace with no new information gained. In a real system you'd also track cost/latency here, since redundant calls are a common source of runaway agent cost.
- **Task completion** — `False`, because even though the agent reached a final answer, its trajectory deviated from the gold path at step 1. This shows why **task success ≠ "did it stop without error"** — you need the full trajectory check, not just presence of a final message.
- **What happens on a wrong action** — nothing in this toy agent *catches* the redundant call mid-loop; it just gets executed and flagged afterward by the eval harness. A more robust agent would validate before calling (e.g., check "have I already fetched this?") so the wrong action never happens, or self-correct by inspecting the trace so far.

## Part B — Mocked: the full 9-metric suite on a realistic scenario

**Scenario:** a customer-support agent with three tools — `search_orders`, `get_refund_policy`, `issue_refund`. User asks: *"My order #4521 arrived broken, can I get a refund?"*

This scenario is deliberately built to exercise every metric in one trace:
- A **typo'd order ID** on the first call → a real tool error
- The agent **retries with the correct ID** → a recovery signal
- A **redundant repeat** of the policy lookup → an unnecessary call
- A real **mutation to the mock "order database"** → something end-state verification can check independently of what the agent claims

**Metrics covered in this part:**
1. Tool Selection Accuracy
2. Tool Call Correctness (Arguments)
3. Unnecessary / Redundant Tool Calls
4. Step-wise Accuracy
5. Task Success Rate
6. Trajectory Match (fuzzy)
7. End-State Verification
8. Recovery / Self-Correction Rate
9. Cost / Efficiency proxy (call count as a stand-in for tokens/latency)

### State schema for this scenario

Defines the LangGraph state for the refund-agent run: `order_db` (the mutable mock database used later for end-state verification), `gold_trajectory` (the tool set the task is expected to exercise), and a `trace` that — unlike Part A — also records `status` (`"ok"`/`"error"`) and `redundant` per step, since those two fields are exactly what the metrics further down key off of.

In [ ]:
# ============ STATE SCHEMA ============
class RefundAgentState(TypedDict):
    query: str
    order_db: Dict[int, Dict]       # mock "system of record" -- what end-state verification checks
    gold_trajectory: List[str]       # gold tool set the task should exercise
    trace: List[Dict]                # [{"step","tool","args","result","status","redundant"}]
    current_step: int
    final_answer: Optional[str]
    max_steps: int

### Tools + mock system of record

`order_db` is a plain dict standing in for a real database. `issue_refund` **mutates** it — this is what lets us check End-State Verification independently of the agent's own final message.

In [ ]:
# ============ TOOLS + MOCK SYSTEM OF RECORD ============
def tool_search_orders(args, db):
    order_id = args.get("order_id")
    order = db.get(order_id)
    if order is None:
        return {"status": "error", "result": f"order {order_id} not found"}
    return {"status": "ok", "result": order}

def tool_get_refund_policy(args, db):
    return {"status": "ok", "result": "Orders damaged on arrival are eligible for full refund."}

def tool_issue_refund(args, db):
    order_id = args.get("order_id")
    amount = args.get("amount")
    order = db.get(order_id)
    if order is None:
        return {"status": "error", "result": f"cannot refund: order {order_id} not found"}
    order["refund_issued"] = True     # <-- the actual state mutation
    order["refund_amount"] = amount
    return {"status": "ok", "result": f"refund of ${amount} issued for order {order_id}"}

TOOLS = {
    "search_orders": tool_search_orders,
    "get_refund_policy": tool_get_refund_policy,
    "issue_refund": tool_issue_refund,
}
print("Tools registered:", list(TOOLS.keys()))

### Mock planner with realistic failure modes

Unlike Part A's simpler plan, this one deliberately includes an error (typo'd order ID) followed by a correction, plus a redundant call — so every metric below has something real to measure.

In [ ]:
# ============ MOCK PLANNER ============
def mock_plan_next_step(state: RefundAgentState) -> Optional[Dict]:
    plan = [
        {"tool": "search_orders", "args": {"order_id": 4251}},         # typo -> will error
        {"tool": "search_orders", "args": {"order_id": 4521}},         # corrected -> recovery
        {"tool": "get_refund_policy", "args": {}},
        {"tool": "get_refund_policy", "args": {}},                      # redundant repeat
        {"tool": "issue_refund", "args": {"order_id": 4521, "amount": 45}},
    ]
    step = state["current_step"]
    return plan[step] if step < len(plan) else None

### Agent loop + graph

`plan_and_act_node` executes the next planned tool call against `order_db`, marks it `redundant` if the identical `(tool, args)` pair already **succeeded** earlier in the trace, and appends the full record — including `status` — to `trace`. `should_continue` loops back to `plan_and_act` until the planner runs out of steps (or `max_steps` is hit), then ends. The graph itself is a single self-looping node, simpler than Part A's two-node design, because evaluation is done entirely separately below rather than as a graph node.

In [ ]:
# ============ AGENT LOOP + GRAPH ============
def plan_and_act_node(state: RefundAgentState) -> RefundAgentState:
    next_call = mock_plan_next_step(state)
    if next_call is None or state["current_step"] >= state["max_steps"]:
        state["final_answer"] = "Your refund of $45 has been issued."
        return state

    tool_name, args = next_call["tool"], next_call["args"]
    outcome = TOOLS[tool_name](args, state["order_db"])

    # A call is redundant if the SAME (tool, args) already succeeded earlier in the trace
    is_duplicate = any(
        t["tool"] == tool_name and t["args"] == args and t["status"] == "ok"
        for t in state["trace"]
    )

    state["trace"].append({
        "step": state["current_step"],
        "tool": tool_name,
        "args": args,
        "result": outcome["result"],
        "status": outcome["status"],      # "ok" or "error"
        "redundant": is_duplicate,
    })
    state["current_step"] += 1
    return state


def should_continue(state: RefundAgentState) -> str:
    return "plan_and_act" if state.get("final_answer") is None else END


builder = StateGraph(RefundAgentState)
builder.add_node("plan_and_act", plan_and_act_node)
builder.set_entry_point("plan_and_act")
builder.add_conditional_edges("plan_and_act", should_continue, {"plan_and_act": "plan_and_act", END: END})
refund_agent_graph = builder.compile()
print("Graph compiled")

### Run the scenario

Seeds `order_db` with one order (`4521`, `refund_issued: False`) and a `gold_trajectory` of the three tools the task should exercise, then invokes the graph. `trace` and `order_db_after` — the two artifacts every metric below is computed from — are pulled out of the result here.

In [ ]:
# ============ RUN THE SCENARIO ============
initial_state: RefundAgentState = {
    "query": "My order #4521 arrived broken, can I get a refund?",
    "order_db": {4521: {"item": "Headphones", "price": 45, "refund_issued": False}},
    "gold_trajectory": ["search_orders", "get_refund_policy", "issue_refund"],
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "max_steps": 6,
}

result = refund_agent_graph.invoke(initial_state)
trace = result["trace"]
order_db_after = result["order_db"]

print("=== TRACE ===")
for t in trace:
    print(f"  step {t['step']}: {t['tool']}({t['args']}) -> {t['result']!r}  [{t['status']}]  redundant={t['redundant']}")

print("\nFinal answer:", result["final_answer"])

**Reading the trace:** step 0 fails (wrong order ID), step 1 corrects it and succeeds, steps 2-3 look up policy twice (only the first was necessary), step 4 issues the refund. This one trace contains an error, a recovery, and a redundancy — exactly the raw material each metric below needs.

### Computing all 9 metrics from the trace

Each metric reads from `trace` and `order_db_after` — nothing here needs a new agent run, which mirrors how a real eval harness works: run once, score many ways.

#### 1. Tool Selection Accuracy

Checks *coverage*, not order: does the set of tools the agent actually called include every tool in the gold set? This is the loosest of the nine metrics — it would pass even if the agent called tools in the wrong order or threw in extra redundant calls, which is exactly why the metrics that follow narrow in on those specific failure modes.

In [ ]:
# 1. Tool Selection Accuracy -- did the agent's trace cover every tool the gold set required?
gold_tools_set = set(result["gold_trajectory"])
called_tools_set = {t["tool"] for t in trace}

tool_selection_coverage = len(gold_tools_set & called_tools_set) / len(gold_tools_set)
print(f"1. Tool Selection Accuracy: {tool_selection_coverage:.2f}   (gold tools = {gold_tools_set})")

#### 2. Tool Call Correctness (Arguments)

Passing metric 1 only confirms the *right tools* were called — this checks whether the *arguments* passed to `issue_refund` were actually valid (correct `order_id`, a positive `amount`). A tool can be correctly selected and still be called with wrong or malformed arguments; this metric catches that separately, on the calls that actually succeeded.

In [ ]:
# 2. Tool Call Correctness (Arguments) -- right tool, but were the ARGS actually right?
issue_calls = [t for t in trace if t["tool"] == "issue_refund" and t["status"] == "ok"]
correct_args = all(
    c["args"].get("order_id") == 4521 and c["args"].get("amount", 0) > 0
    for c in issue_calls
)
print(f"2. Tool Call Correctness (issue_refund args valid): {correct_args}")

#### 3. Unnecessary / Redundant Tool Calls

Simply counts the trace entries `plan_and_act_node` already flagged `redundant` — a call whose `(tool, args)` pair had already **succeeded** earlier in the same trace, adding no new information. Here that's the duplicate `get_refund_policy` lookup.

In [ ]:
# 3. Unnecessary / Redundant Tool Calls -- same (tool, args) succeeding more than once
redundant_calls = [t for t in trace if t["redundant"]]
print(f"3. Unnecessary Tool Calls: {len(redundant_calls)}  -> {[t['tool'] for t in redundant_calls]}")

#### 4. Step-wise Accuracy

Fraction of *all* trace steps that were both `status == "ok"` and not `redundant` — i.e. steps that made real forward progress. Unlike Tool Selection Accuracy (which only checks coverage of the tool set), this penalizes wasted steps — errors and redundant calls — at the individual-step level.

In [ ]:
# 4. Step-wise Accuracy -- fraction of steps that were both successful AND non-redundant
good_steps = [t for t in trace if t["status"] == "ok" and not t["redundant"]]
step_accuracy = len(good_steps) / len(trace)
print(f"4. Step-wise Accuracy: {step_accuracy:.2f}   ({len(good_steps)}/{len(trace)} steps)")

#### 5. Task Success Rate

For this single run, checks the ground-truth outcome directly: was `refund_issued` actually set, and does `refund_amount` match the expected `45`? In a real eval suite this boolean would be computed per task and averaged across many tasks to get an actual *rate* — here it's just the one-task building block.

In [ ]:
# 5. Task Success Rate -- for THIS task, did the intended outcome actually happen?
# (In a real eval you'd average this boolean across many tasks to get a rate.)
task_success = order_db_after[4521]["refund_issued"] and order_db_after[4521]["refund_amount"] == 45
print(f"5. Task Success (this task): {task_success}")

#### 6. Trajectory Match (fuzzy / set-based)

Compares the *set* of tools that ended in `"ok"` against the gold tool set, ignoring order, retries, and errors entirely. Deliberately looser than an exact-sequence match, which would fail here purely because of the typo'd first call and the redundant lookup — even though the agent still did the right things overall.

In [ ]:
# 6. Trajectory Match (fuzzy / set-based) -- ignores order and retries, checks the
# set of SUCCESSFUL tool types matches the gold set. A stricter exact-sequence
# match would fail here because of the extra error + redundant steps.
actual_ok_tools = {t["tool"] for t in trace if t["status"] == "ok"}
trajectory_fuzzy_match = actual_ok_tools == gold_tools_set
print(f"6. Trajectory Match (fuzzy, set-based): {trajectory_fuzzy_match}")

#### 7. End-State Verification

The most important check in this suite: instead of trusting `result['final_answer']` — a string the agent generated, which could be wrong or fabricated — this reads `order_db_after`, the actual mutated "system of record," directly, to confirm the refund really was recorded and not just claimed.

In [ ]:
# 7. End-State Verification -- check the actual "system of record" directly,
# NOT the agent's self-reported final_answer string. This is the check that would
# catch a case where the agent claims success but the mutation never happened.
end_state_verified = (
    order_db_after.get(4521, {}).get("refund_issued") is True
    and order_db_after.get(4521, {}).get("refund_amount") == 45
)
print(f"7. End-State Verification (checked DB directly): {end_state_verified}")
print(f"   Agent claimed: {result['final_answer']!r}")
print(f"   DB actually shows: {order_db_after[4521]}")

#### 8. Recovery / Self-Correction Rate

For every trace entry with `status == "error"`, checks whether a *later* entry called the *same tool* and succeeded — that's the signal the agent noticed its own mistake and corrected it, rather than giving up or blindly repeating the same bad call. Reported as `recovered / total errors`, or `None` when there were no errors to recover from.

In [ ]:
# 8. Recovery / Self-Correction Rate -- for each error, was there a LATER successful
# call to the SAME tool? That's the signal the agent noticed and corrected itself.
errors = [t for t in trace if t["status"] == "error"]
recovered = sum(
    1 for err in errors
    if any(t["step"] > err["step"] and t["tool"] == err["tool"] and t["status"] == "ok" for t in trace)
)
recovery_rate = recovered / len(errors) if errors else None
print(f"8. Recovery Rate: {recovered}/{len(errors)} errors recovered  -> {recovery_rate}")

#### 9. Cost / Efficiency proxy

Treats redundant calls and errors as "wasted" and computes `1 - wasted/total` as a lightweight stand-in for a real cost metric (token usage, latency) that in production you'd pull from tracing infrastructure instead — see the Databricks mapping table below for where that data would actually come from.

In [ ]:
# 9. Cost / Efficiency proxy -- in production this would be tokens/latency (e.g. from
# MLflow Tracing span metadata); call count is a simple stand-in for the same idea.
total_calls = len(trace)
wasted_calls = len(redundant_calls) + len(errors)
efficiency = 1 - (wasted_calls / total_calls)
print(f"9. Cost/Efficiency proxy: {total_calls} calls total, {wasted_calls} wasted (error+redundant)  -> efficiency={efficiency:.2f}")

**Reading the results together:**

- Tool selection and end-state verification both come back **positive** — the agent used every tool it needed to and the refund really was recorded in the "database," not just claimed.
- But step-wise accuracy is only **0.60** and cost efficiency only **0.60** — because 2 of the 5 steps were waste (1 error, 1 redundant call). This is exactly why **outcome metrics alone can hide inefficiency**: the task succeeded, but it took 40% more calls than necessary to get there.
- Recovery rate of **1.0** is a genuinely good sign — the one error that occurred was caught and corrected, not left to fail silently or retried blindly with the same bad input.
- This is the core argument for running **step-level and trajectory-level metrics together**: task success alone would have reported "pass" and told you nothing about the wasted policy lookup or the initial wrong order ID.

### Databricks implementation notes for this scenario

| Metric | How this maps onto Databricks |
|---|---|
| Tool Selection Accuracy | Gold tool set stored in a Delta table; compared against tool spans captured by MLflow Tracing via `mlflow.langchain.autolog()` |
| Tool Call Correctness | `search_orders` / `issue_refund` defined as **Unity Catalog Functions** with typed signatures (`order_id: INT`) — a malformed call fails validation before it ever executes |
| Unnecessary Tool Calls | Query the MLflow trace table for duplicate `(tool, inputs)` spans within the same `request_id` |
| Step-wise Accuracy | Custom per-step scorer passed into `mlflow.genai.evaluate()`, iterating trace spans |
| Task Success Rate | Eval set of `(ticket_id, expected_outcome)` in Delta; agent run via a Databricks Job; pass/fail logged as an MLflow metric, trended in a Lakeview dashboard |
| Trajectory Match | Gold trajectories as JSON in Delta; a Unity Catalog-registered comparison function invoked as a custom `mlflow.genai.evaluate()` metric |
| End-State Verification | Delta Lake time-travel diff (`VERSION AS OF`) comparing the `order_db`-equivalent table before and after the agent run, inside the same Databricks Workflow |
| Recovery Rate | MLflow Tracing marks error-status spans automatically; a scheduled notebook pattern-matches them against later successful spans in the same trace tree |
| Cost / Efficiency | MLflow Tracing captures token usage and latency **per span automatically** — no manual instrumentation needed, just query the trace table |

Databricks now recommends **MLflow 3's `mlflow.genai.evaluate()`** with built-in LLM judges (Mosaic AI Agent Evaluation) over the older `mlflow.evaluate()` API — the pattern above uses that newer surface. This is also the same `mlflow.genai.evaluate()` surface Module 2.4 used directly, from the DeepEval side of this tutorial rather than the Databricks-specific angle.

## Part C — Real: a live tool-calling agent

This time the planner is a real `ChatOpenAI` model with tools bound via `bind_tools` — the model itself decides which tool to call, with what arguments, and when it's done. The graph loops `agent -> tools -> agent -> ...` until the model responds without requesting a tool call, then hands the resulting trace to a separate `evaluate` node that scores it against a gold trajectory — the same "agent never grades itself" separation from Part A, now with a real model driving the loop instead of a scripted mock planner.

In [ ]:
# ============ REAL LLM SETUP ============
import os
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY (shell env or .env) before running this cell."

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("OpenAI client ready")

In [ ]:
# ============ REAL TOOLS THE AGENT CAN CALL ============
@tool
def search_docs(topic: str) -> str:
    """Search internal docs for information about a topic."""
    return f"docs about {topic}: BST delete runs in O(h) time where h is tree height."


@tool
def calculator(expr: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '2**10'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expr) <= allowed:
        return "error: expression contains disallowed characters"
    return str(eval(expr, {"__builtins__": {}}, {}))


@tool
def send_email(to: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"email sent to {to}"


TOOLS = [search_docs, calculator, send_email]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
agent_llm = llm.bind_tools(TOOLS)

print("Tools registered:", list(TOOLS_BY_NAME.keys()))

In [ ]:
# ============ STATE SCHEMA ============
class AgentEvalState(TypedDict):
    query: str
    gold_tool_sequence: List[str]     # ground truth trajectory, defined by the eval harness
    messages: List                    # the real conversation, incl. tool calls/results
    trace: List[Dict]                 # [{"tool", "args", "result", "used"}]
    final_answer: Optional[str]
    task_success: Optional[bool]
    tool_selection_correct: List[bool]
    unnecessary_calls: List[str]

In [ ]:
# ============ GRAPH NODES ============
MAX_STEPS = 6  # safety cap so a misbehaving agent can't loop forever


def agent_node(state: AgentEvalState) -> AgentEvalState:
    if not state["messages"]:
        state["messages"] = [
            SystemMessage("You are a helpful assistant. Use tools when needed to answer the "
                          "user's request, then give a final natural-language answer. Do not "
                          "call the same tool with the same arguments twice."),
            HumanMessage(state["query"]),
        ]

    response: AIMessage = agent_llm.invoke(state["messages"])
    state["messages"] = state["messages"] + [response]

    if not response.tool_calls:
        state["final_answer"] = response.content

    return state


def tools_node(state: AgentEvalState) -> AgentEvalState:
    last_message: AIMessage = state["messages"][-1]
    tool_messages = []

    for call in last_message.tool_calls:
        tool_name, args = call["name"], call["args"]

        # A call is "unnecessary" if the exact same (tool, args) already ran --
        # the result would be identical, so it added no new information.
        is_duplicate = any(t["tool"] == tool_name and t["args"] == args for t in state["trace"])

        result = TOOLS_BY_NAME[tool_name].invoke(args)
        state["trace"].append({
            "tool": tool_name, "args": args, "result": result, "used": not is_duplicate,
        })
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    state["messages"] = state["messages"] + tool_messages
    return state


def should_continue(state: AgentEvalState) -> str:
    if state.get("final_answer") is not None:
        return "evaluate"
    if len(state["trace"]) >= MAX_STEPS:
        state["final_answer"] = "Task stopped: exceeded max tool-call budget."
        return "evaluate"
    return "tools"


def evaluate_node(state: AgentEvalState) -> AgentEvalState:
    actual_tools = [t["tool"] for t in state["trace"]]
    gold_tools = state["gold_tool_sequence"]

    # Step-wise trajectory match against the gold sequence -- this is how you
    # evaluate MULTI-STEP tasks: score each step, not just the final answer.
    correctness = [
        (actual_tools[i] if i < len(actual_tools) else None) == gold_tools[i]
        for i in range(len(gold_tools))
    ]
    state["tool_selection_correct"] = correctness

    # Flags raised during the trace itself -- redundant calls with no new info.
    state["unnecessary_calls"] = [t["tool"] for t in state["trace"] if not t["used"]]

    # Task completion = reached a final answer AND every gold step was matched.
    state["task_success"] = (
        state["final_answer"] is not None and sum(correctness) == len(gold_tools)
    )
    return state


print("Nodes defined (agent_node calls the real OpenAI model)")

In [ ]:
# ============ BUILD & RUN THE GRAPH ============
builder = StateGraph(AgentEvalState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "evaluate": "evaluate"})
builder.add_edge("tools", "agent")
builder.add_edge("evaluate", END)

agent_eval_graph = builder.compile()

initial_state: AgentEvalState = {
    "query": "Look up BST delete complexity and compute 2^10",
    "gold_tool_sequence": ["search_docs", "calculator"],   # the IDEAL 2-step trajectory
    "messages": [],
    "trace": [],
    "final_answer": None,
    "task_success": None,
    "tool_selection_correct": [],
    "unnecessary_calls": [],
}

result = agent_eval_graph.invoke(initial_state)

print("=== Trace ===")
for i, step in enumerate(result["trace"]):
    print(f"  Step {i}: {step['tool']}({step['args']}) -> {step['result']!r}  | used={step['used']}")

print()
print("=== Eval ===")
print("Tool selection correctness vs gold:", result["tool_selection_correct"])
print("Unnecessary calls detected:        ", result["unnecessary_calls"])
print("Task success:                      ", result["task_success"])
print("Final answer:                      ", result["final_answer"])

**Reading the output:** unlike Part A/B (which hard-coded a redundant call or a typo'd ID to *force* the demo), a real, reasonably capable model will usually pick the correct minimal trajectory (`search_docs` then `calculator`) and avoid duplicate calls on its own. That itself is worth stating out loud: mocked agent demos need synthetic failures injected to have something to catch, but a real deployed agent needs a harness like this `evaluate` node running continuously in production to catch the cases where the model *doesn't* behave well — e.g. try changing the query to something ambiguous, or add a fourth tool the model might reach for unnecessarily, and re-run this cell to see the eval flag it.

## Summary

| Failure mode | Caught by |
|---|---|
| Wrong tool called, or a tool missing entirely | Tool Selection Accuracy (coverage) |
| Right tool, wrong/malformed arguments | Tool Call Correctness (Arguments) |
| Same call repeated for no new information | Unnecessary / Redundant Tool Calls |
| Wasted steps dragging down an otherwise-fine outcome | Step-wise Accuracy |
| The task's real-world outcome, checked directly | Task Success Rate + End-State Verification |
| A wrong sequence that still hit the right tools overall | Trajectory Match (fuzzy) vs. exact-sequence matching |
| An error the agent noticed and fixed vs. one it didn't | Recovery / Self-Correction Rate |
| Excess tool calls as a stand-in for token/latency cost | Cost / Efficiency proxy |

The throughline across all nine: **task success alone is not enough.** A trace can reach the right final answer while still being inefficient, error-prone, or lucky rather than reliable — which is exactly why this module runs step-level and outcome-level metrics together rather than picking one. Module 4 takes this same idea into a production setting with real tracing infrastructure (Arize Phoenix) instead of a hand-rolled `trace` list; Module 5's capstone applies `TaskCompletionMetric` — the single-outcome metric from Module 2.3 — to a real running multi-agent system, closing the loop between this module's trace-level detail and Module 2's simpler outcome check.